## Common Imports in Malicious Packages

In [8]:
import ast
import re
import sys
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=SyntaxWarning)

# ============================================================
# Configuration
# ============================================================

SETUP_COL = "setup.py"

INPUT_FILE = Path(
    r"D:/12. RQ2 eDySec/Adversarial Attacks/"
    r"MaliciousPackagesDetailsFromFiles.xlsx"
)

OUTPUT_DIR = INPUT_FILE.parent / "import_frequency_figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 20


# ============================================================
# Standard-library classification
# ============================================================

# Python's official standard-library module list.
# Manual additions support environments where deprecated or
# platform-specific modules may not appear.
STDLIB = set(sys.stdlib_module_names)

STDLIB.update(
    {
        "distutils",
        "atexit",
        "fcntl",
        "site",
        "pwd",
        "termios",
        "winreg",
        "msvcrt",
    }
)


# ============================================================
# Matplotlib configuration
# ============================================================

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": [
            "Times New Roman",
            "Times",
            "DejaVu Serif",
        ],
        "font.size": 8,
        "axes.labelsize": 8.5,
        "axes.titlesize": 9,
        "xtick.labelsize": 7.5,
        "ytick.labelsize": 7.5,
        "legend.fontsize": 7.5,
        "axes.linewidth": 0.8,
        "lines.linewidth": 1.2,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


# ============================================================
# Import extraction
# ============================================================

def top_level(name: str) -> str:
    """
    Convert a dotted module name to its top-level package.

    Examples
    --------
    requests.sessions -> requests
    urllib.request     -> urllib
    """
    return name.split(".")[0] if name else name


def imports_from_source(src: str) -> list[str]:
    """
    Extract top-level imported module names from one setup.py body.

    The returned list preserves repeated import occurrences. Therefore:

        import os
        import os

    returns:

        ["os", "os"]

    Static AST parsing is attempted first. A regex fallback is used for
    malformed, truncated, or syntactically invalid source code.

    The source code is never executed.
    """
    modules: list[str] = []

    if not isinstance(src, str) or not src.strip():
        return modules

    # --------------------------------------------------------
    # Primary method: AST parsing
    # --------------------------------------------------------
    try:
        tree = ast.parse(src)

        for node in ast.walk(tree):

            # Handles:
            # import os
            # import os, sys
            # import requests.sessions
            if isinstance(node, ast.Import):
                for alias in node.names:
                    module = top_level(alias.name)

                    if module:
                        modules.append(module)

            # Handles:
            # from urllib import request
            # from pathlib import Path
            #
            # Relative imports are excluded because they do not
            # identify an external top-level module.
            elif isinstance(node, ast.ImportFrom):
                if (
                    node.level == 0
                    and node.module
                ):
                    module = top_level(node.module)

                    if module:
                        modules.append(module)

        return modules

    except (SyntaxError, ValueError, TypeError):
        pass

    # --------------------------------------------------------
    # Fallback method: regex parsing
    # --------------------------------------------------------

    # Handles:
    # import os
    # import os, sys
    # import requests as req
    import_pattern = re.compile(
        r"^\s*import\s+([^#;\n]+)",
        flags=re.MULTILINE,
    )

    for match in import_pattern.finditer(src):
        import_expression = match.group(1)

        for item in import_expression.split(","):
            item = item.strip()

            if not item:
                continue

            # Remove "as alias".
            module_name = re.split(
                r"\s+as\s+",
                item,
                maxsplit=1,
            )[0].strip()

            # Retain only syntactically plausible module names.
            if re.fullmatch(
                r"[A-Za-z_][\w.]*",
                module_name,
            ):
                modules.append(top_level(module_name))

    # Handles:
    # from urllib import request
    # from pathlib import Path
    from_pattern = re.compile(
        r"^\s*from\s+([A-Za-z_][\w.]*)\s+import\b",
        flags=re.MULTILINE,
    )

    for match in from_pattern.finditer(src):
        module_name = match.group(1)

        # Ignore relative imports beginning with a dot.
        if not module_name.startswith("."):
            modules.append(top_level(module_name))

    return modules


# ============================================================
# Figure-saving utility
# ============================================================

def save_figure(
    fig: plt.Figure,
    filename: str,
) -> None:
    """
    Save one figure in vector and high-resolution raster formats.
    """
    formats = {
        "pdf": {},
        "svg": {},
        "png": {"dpi": 600},
    }

    for extension, options in formats.items():
        output_path = OUTPUT_DIR / f"{filename}.{extension}"

        fig.savefig(
            output_path,
            bbox_inches="tight",
            pad_inches=0.03,
            **options,
        )

    plt.close(fig)


# ============================================================
# Figure 1: Top import prevalence
# ============================================================

def plot_top_import_prevalence(
    ranking: pd.DataFrame,
    n_packages: int,
    top_k: int = 20,
) -> None:
    """
    Produce a single-column horizontal bar chart showing the most
    frequently imported modules.
    """
    data = ranking.head(top_k).copy()

    data["share"] = (
        100.0 * data["package_count"] / n_packages
    )

    data["type_code"] = np.where(
        data["kind"].eq("stdlib"),
        "S",
        "T",
    )

    data["display_name"] = (
        data["module"]
        + " ("
        + data["type_code"]
        + ")"
    )

    # Reverse order so the highest-ranked module appears at the top.
    data = data.iloc[::-1].reset_index(drop=True)

    fig, ax = plt.subplots(
        figsize=(3.35, 4.75)
    )

    bars = ax.barh(
        data["display_name"],
        data["share"],
        height=0.68,
    )

    # Add a hatch to third-party/other modules.
    for bar, kind in zip(bars, data["kind"]):
        if kind != "stdlib":
            bar.set_hatch("///")

    max_share = data["share"].max()

    ax.set_xlim(
        0,
        np.ceil((max_share + 8) / 10) * 10,
    )

    ax.set_xlabel(
        "Packages importing module (%)"
    )

    ax.set_ylabel("")

    ax.grid(
        axis="x",
        linewidth=0.45,
        alpha=0.30,
    )

    ax.set_axisbelow(True)

    # Bar-end labels.
    for bar, count, share in zip(
        bars,
        data["package_count"],
        data["share"],
    ):
        ax.text(
            share + 0.8,
            bar.get_y() + bar.get_height() / 2,
            f"{share:.1f}% ({count:,})",
            va="center",
            ha="left",
            fontsize=6.5,
        )

    ax.text(
        0,
        -0.105,
        (
            f"N = {n_packages:,} setup.py bodies. "
            "S: standard library; T: third-party/other. "
            "Imports may co-occur."
        ),
        transform=ax.transAxes,
        fontsize=6.3,
        va="top",
    )

    fig.subplots_adjust(
        left=0.30,
        right=0.98,
        top=0.99,
        bottom=0.13,
    )

    save_figure(
        fig,
        "figure_top20_import_prevalence",
    )


# ============================================================
# Figure 2: Rank-frequency distribution
# ============================================================

def plot_rank_frequency(
    ranking: pd.DataFrame,
    n_packages: int,
) -> None:
    """
    Show the complete rank-frequency distribution on a logarithmic
    y-axis. This figure communicates the concentrated head and long
    tail of setup.py imports.
    """
    data = ranking.copy()

    data["rank"] = np.arange(
        1,
        len(data) + 1,
    )

    data["share"] = (
        100.0 * data["package_count"] / n_packages
    )

    fig, ax = plt.subplots(
        figsize=(7.0, 2.85)
    )

    ax.plot(
        data["rank"],
        data["share"],
        marker="o",
        markersize=3.0,
        markeredgewidth=0.4,
    )

    ax.fill_between(
        data["rank"],
        data["share"],
        alpha=0.10,
    )

    # Distinguish the top-10 head from the remaining tail.
    ax.axvline(
        10.5,
        linestyle="--",
        linewidth=0.8,
        alpha=0.70,
    )

    ax.set_yscale("log")

    ax.set_xlabel("Import rank")

    ax.set_ylabel(
        "Packages importing module (%)"
    )

    ax.set_xlim(
        1,
        len(data),
    )

    positive_shares = data.loc[
        data["share"] > 0,
        "share",
    ]

    ax.set_ylim(
        positive_shares.min() * 0.70,
        positive_shares.max() * 1.60,
    )

    tick_candidates = [
        1,
        5,
        10,
        20,
        30,
        40,
        50,
        60,
        len(data),
    ]

    ticks = sorted(
        {
            tick
            for tick in tick_candidates
            if tick <= len(data)
        }
    )

    ax.set_xticks(ticks)

    ax.grid(
        axis="both",
        linewidth=0.45,
        alpha=0.27,
    )

    ax.set_axisbelow(True)

    # Region labels.
    ax.text(
        5.5,
        0.96,
        "high-frequency head",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=7,
    )

    tail_midpoint = (
        10.5 + len(data)
    ) / 2

    ax.text(
        tail_midpoint,
        0.96,
        "long tail",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=7,
    )

    # Label selected modules without overcrowding the plot.
    selected_ranks = [
        1,
        2,
        3,
        4,
        6,
        10,
        20,
        30,
        40,
        60,
        len(data),
    ]

    selected_ranks = sorted(
        {
            rank
            for rank in selected_ranks
            if rank <= len(data)
        }
    )

    for index, rank in enumerate(selected_ranks):
        row = data.iloc[rank - 1]

        vertical_offset = (
            7 if index % 2 == 0 else -11
        )

        horizontal_alignment = (
            "right"
            if rank == len(data)
            else "left"
        )

        horizontal_offset = (
            -4
            if rank == len(data)
            else 5
        )

        ax.annotate(
            row["module"],
            xy=(
                row["rank"],
                row["share"],
            ),
            xytext=(
                horizontal_offset,
                vertical_offset,
            ),
            textcoords="offset points",
            fontsize=6.7,
            ha=horizontal_alignment,
            va=(
                "bottom"
                if vertical_offset > 0
                else "top"
            ),
        )

    ax.text(
        0,
        -0.22,
        (
            "Logarithmic y-axis. Frequency denotes prevalence "
            "and should not be interpreted as maliciousness."
        ),
        transform=ax.transAxes,
        fontsize=6.5,
        va="top",
    )

    fig.subplots_adjust(
        left=0.10,
        right=0.99,
        top=0.98,
        bottom=0.27,
    )

    save_figure(
        fig,
        "figure_import_rank_frequency",
    )


# ============================================================
# Figure 3: Cumulative import mass
# ============================================================

def plot_cumulative_import_mass(
    ranking: pd.DataFrame,
) -> None:
    """
    Plot the cumulative percentage of package-module incidences
    represented by the ranked imports.

    This is not the percentage of unique packages because packages
    may import multiple modules.
    """
    data = ranking.copy()

    data["rank"] = np.arange(
        1,
        len(data) + 1,
    )

    total_package_module_incidences = (
        data["package_count"].sum()
    )

    data["cumulative_share"] = (
        100.0
        * data["package_count"].cumsum()
        / total_package_module_incidences
    )

    fig, ax = plt.subplots(
        figsize=(7.0, 2.70)
    )

    ax.plot(
        data["rank"],
        data["cumulative_share"],
        marker="o",
        markersize=2.8,
        markevery=max(
            1,
            len(data) // 15,
        ),
    )

    ax.fill_between(
        data["rank"],
        data["cumulative_share"],
        alpha=0.10,
    )

    ax.set_xlabel(
        "Number of highest-ranked imports included"
    )

    ax.set_ylabel(
        "Cumulative import mass (%)"
    )

    ax.set_xlim(
        1,
        len(data),
    )

    ax.set_ylim(
        0,
        102,
    )

    ax.set_yticks(
        [0, 20, 40, 60, 80, 100]
    )

    tick_candidates = [
        1,
        10,
        20,
        30,
        40,
        50,
        60,
        len(data),
    ]

    ticks = sorted(
        {
            tick
            for tick in tick_candidates
            if tick <= len(data)
        }
    )

    ax.set_xticks(ticks)

    ax.grid(
        axis="both",
        linewidth=0.45,
        alpha=0.27,
    )

    ax.set_axisbelow(True)

    # Identify ranks required to explain 80%, 90%, and 95%.
    thresholds = [80, 90, 95]

    for index, threshold in enumerate(thresholds):
        matching = data.loc[
            data["cumulative_share"] >= threshold
        ]

        if matching.empty:
            continue

        row = matching.iloc[0]

        ax.scatter(
            row["rank"],
            row["cumulative_share"],
            s=24,
            zorder=3,
        )

        offset = (
            (6, 8)
            if index % 2 == 0
            else (6, -14)
        )

        ax.annotate(
            (
                f"{threshold}% at rank "
                f"{int(row['rank'])}"
            ),
            xy=(
                row["rank"],
                row["cumulative_share"],
            ),
            xytext=offset,
            textcoords="offset points",
            fontsize=6.8,
            va=(
                "bottom"
                if offset[1] > 0
                else "top"
            ),
        )

    # Annotate cumulative coverage for standard top-k cutoffs.
    for rank in [10, 20, 40, 60]:
        if rank > len(data):
            continue

        row = data.iloc[rank - 1]

        ax.scatter(
            row["rank"],
            row["cumulative_share"],
            s=18,
            zorder=3,
        )

        ax.annotate(
            f"Top {rank}: {row['cumulative_share']:.1f}%",
            xy=(
                row["rank"],
                row["cumulative_share"],
            ),
            xytext=(5, -13),
            textcoords="offset points",
            fontsize=6.4,
            va="top",
        )

    ax.text(
        0,
        -0.23,
        (
            "Import mass is calculated from package-level module "
            "incidences; one package may contribute to several modules."
        ),
        transform=ax.transAxes,
        fontsize=6.5,
        va="top",
    )

    fig.subplots_adjust(
        left=0.10,
        right=0.99,
        top=0.98,
        bottom=0.29,
    )

    save_figure(
        fig,
        "figure_cumulative_import_mass",
    )


# ============================================================
# Main analysis
# ============================================================

def main(path: Path) -> None:
    """
    Extract import statistics, save the full ranking, and generate
    publication-ready figures.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Input file was not found:\n{path}"
        )

    df = pd.read_excel(
        path,
        usecols=[SETUP_COL],
    )

    cells = df[SETUP_COL].dropna()

    # Number of packages importing each module.
    # Each package contributes at most one vote per module.
    document_frequency = Counter()

    # Total syntactic import occurrences across all packages.
    total_occurrences = Counter()

    packages_with_imports = 0

    for source in cells:
        imported_modules = imports_from_source(
            source
        )

        if not imported_modules:
            continue

        packages_with_imports += 1

        # One package-level vote per module.
        document_frequency.update(
            set(imported_modules)
        )

        # All observed syntactic occurrences.
        total_occurrences.update(
            imported_modules
        )

    n_packages = len(cells)

    print(
        f"Rows with a setup.py body : "
        f"{n_packages}"
    )

    print(
        f"Rows with >=1 import      : "
        f"{packages_with_imports}"
    )

    print(
        f"Distinct top-level imports: "
        f"{len(document_frequency)}\n"
    )

    # --------------------------------------------------------
    # Console results
    # --------------------------------------------------------
    for k in [10, 20, 30, 40, 50, 60]:
        print(
            f"===== TOP {k} imports "
            f"(by # of packages) ====="
        )

        print(
            f"{'rank':>4}  "
            f"{'module':<22}"
            f"{'pkgs':>7}"
            f"{'share':>9}  "
            f"kind"
        )

        for rank, (module, count) in enumerate(
            document_frequency.most_common(k),
            start=1,
        ):
            kind = (
                "stdlib"
                if module in STDLIB
                else "3rd-party/other"
            )

            share = (
                100.0 * count / n_packages
            )

            print(
                f"{rank:>4}  "
                f"{module:<22}"
                f"{count:>7}"
                f"{share:>8.1f}%  "
                f"{kind}"
            )

        print()

    # --------------------------------------------------------
    # Build full ranking dataframe
    # --------------------------------------------------------
    ranking = pd.DataFrame(
        [
            {
                "module": module,
                "package_count": count,
                "package_share_percent": (
                    100.0 * count / n_packages
                ),
                "total_occurrences": (
                    total_occurrences[module]
                ),
                "mean_occurrences_per_importing_package": (
                    total_occurrences[module] / count
                ),
                "kind": (
                    "stdlib"
                    if module in STDLIB
                    else "3rd-party/other"
                ),
            }
            for module, count
            in document_frequency.most_common()
        ]
    )

    ranking.insert(
        0,
        "rank",
        np.arange(
            1,
            len(ranking) + 1,
        ),
    )

    csv_path = (
        OUTPUT_DIR
        / "import_frequency_ranking.csv"
    )

    ranking.to_csv(
        csv_path,
        index=False,
    )

    print(
        f"Full ranking written to:\n"
        f"{csv_path}\n"
    )

    # --------------------------------------------------------
    # Generate figures
    # --------------------------------------------------------
    plot_top_import_prevalence(
        ranking=ranking,
        n_packages=n_packages,
        top_k=TOP_K,
    )

    plot_rank_frequency(
        ranking=ranking,
        n_packages=n_packages,
    )

    plot_cumulative_import_mass(
        ranking=ranking,
    )

    print(
        "Figures generated successfully:"
    )

    print(
        "  1. figure_top20_import_prevalence"
    )

    print(
        "  2. figure_import_rank_frequency"
    )

    print(
        "  3. figure_cumulative_import_mass"
    )

    print(
        f"\nOutput directory:\n{OUTPUT_DIR}"
    )


if __name__ == "__main__":
    main(INPUT_FILE)

Rows with a setup.py body : 6480
Rows with >=1 import      : 6432
Distinct top-level imports: 106

===== TOP 10 imports (by # of packages) =====
rank  module                   pkgs    share  kind
   1  os                       4711    72.7%  stdlib
   2  distutils                3684    56.9%  stdlib
   3  subprocess               2981    46.0%  stdlib
   4  setuptools               2814    43.4%  3rd-party/other
   5  sys                      1633    25.2%  stdlib
   6  requests                 1379    21.3%  3rd-party/other
   7  random                   1039    16.0%  stdlib
   8  pip                       910    14.0%  3rd-party/other
   9  win32com                  909    14.0%  3rd-party/other
  10  fernet                    745    11.5%  3rd-party/other

===== TOP 20 imports (by # of packages) =====
rank  module                   pkgs    share  kind
   1  os                       4711    72.7%  stdlib
   2  distutils                3684    56.9%  stdlib
   3  subprocess         